### 목적에 맞는 예제 선택기

In [ ]:
# ===== 패키지 설치 (최초 1회만 실행) =====
!pip install python-dotenv
!pip install -U langchain langchain-openai langchain-teddynote

# ===== 필요한 모듈 import =====
import os
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, load_prompt
from datetime import datetime

# ===== 환경변수(.env) 로드 =====
load_dotenv()  # override=False가 기본값이므로 시스템 환경변수가 우선순위를 가짐

# (선택) 정상 로드 확인
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

# ===== LangSmith 추적 시작 =====
logging.langsmith("CH02-Prompt")  # 프로젝트명 입력

# ===== LLM 객체 생성 =====
llm = ChatOpenAI()

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_teddynote.prompts import CustomExampleSelector

examples = [
    {
        "instruction": "당신은 회의록 작성 전문가 입니다...",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다...",
        "answer": """...""",
    },
    {
        "instruction": "당신은 요약 전문가 입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량...",
        "answer": """문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서...""",
    },
    {
        "instruction": "당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요",
        "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다...",
        "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, ...",
    },
]

# 커스텀 예제 선택기 생성
custom_selector = CustomExampleSelector(examples, OpenAIEmbeddings())

# 커스텀 예제 선택기를 사용했을 때 결과
custom_selector.select_examples({"instruction": "다음 문장으로 회의록을 작성해 주세요"})

[{'instruction': '당신은 문장 교정 전문가 입니다. 다음 주어진 문장을 교정해 주세요',
  'input': '우리 회사는 새로운 마케팅 전략을 도입하려고 한다...',
  'answer': '본 회사는 새로운 마케팅 전략을 도입함으로써, ...'}]

In [6]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{instruction}:\n{input}"),
        ("ai", "{answer}"),
    ]
)

custom_fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=custom_selector, # 커스텀 예제 선택기 사용
    example_prompt=example_prompt, # 예제 프롬프트 사용
)

custom_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.",
        ),
        custom_fewshot_prompt,
        ("human", "{instruction}\n{input}"),
    ]
)

In [ ]:
from langchain_teddynote.messages import stream_response

chain = custom_prompt | llm # 체인을 생성

question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다...",
}

stream_response(chain.stream(question)) # 실행 및 결과 출력